# บทที่ 11: พื้นฐาน SQL สำหรับเชื่อมข้อมูลเพื่อการวิเคราะห์

ในการวิเคราะห์ข้อมูลจริง ข้อมูลที่ต้องใช้อาจกระจายอยู่ในหลายตาราง หลายไฟล์ หรือหลายหน่วยงาน เช่น

- ข้อมูลบุคคล
- ข้อมูลครัวเรือน
- ข้อมูลพื้นที่
- ข้อมูลรายได้และหนี้สิน
- ข้อมูลการให้บริการ
- ข้อมูลจากระบบงานต่างกัน

หากต้องการตอบคำถามที่ใช้ข้อมูลจากหลายแหล่ง เราต้องเลือก สรุป และเชื่อมข้อมูลเหล่านั้นเข้าด้วยกัน

ในบทนี้ เราจะใช้ **SQL** ผ่าน **DuckDB** บน Python เพื่อ Query ข้อมูลที่อยู่ใน pandas DataFrame โดยไม่ต้องติดตั้งระบบฐานข้อมูลแยกต่างหาก

จุดสำคัญของบทนี้คือ

> ก่อนเชื่อมข้อมูล ต้องเข้าใจ Grain ของแต่ละตาราง ทำ Key ให้สอดคล้องกัน และตรวจสอบผลหลัง JOIN เสมอ

## ผลการเรียนรู้ที่คาดหวัง

เมื่อเรียนจบบทนี้ ผู้เรียนจะสามารถ

1. ใช้ DuckDB Query ข้อมูลจาก pandas DataFrame ได้
2. ใช้ SQL เพื่อเลือก กรอง เรียง และสรุปข้อมูลได้
3. เลือกและใช้ JOIN ให้เหมาะกับคำถามวิเคราะห์ได้
4. สร้าง Query ที่ใช้ซ้ำได้ด้วย CTE และ Python Function

## ลำดับเนื้อหา

บทเรียนนี้ประกอบด้วยหัวข้อต่อไปนี้

1. แนวคิดการใช้ SQL กับงานวิเคราะห์ข้อมูล
2. เตรียม Library และข้อมูลตัวอย่าง
3. ตรวจสอบและเตรียม DataFrame
4. สร้าง DuckDB Connection
5. Register DataFrame เป็นตาราง
6. เลือกข้อมูลด้วย `SELECT`
7. กรองข้อมูลด้วย `WHERE`
8. เรียงและจำกัดผลลัพธ์ด้วย `ORDER BY` และ `LIMIT`
9. สร้างเงื่อนไขด้วย `CASE WHEN`
10. สรุปข้อมูลด้วย Aggregate Function
11. จัดกลุ่มด้วย `GROUP BY` และ `HAVING`
12. ทำความเข้าใจ Grain และ Key
13. แนวคิดและประเภทของ JOIN
14. สรุปข้อมูลก่อน JOIN
15. ใช้ `INNER JOIN`, `LEFT JOIN` และ `FULL OUTER JOIN`
16. จัดการ `NULL` ด้วย `COALESCE`
17. จัด Query ด้วย CTE
18. ตรวจสอบผลลัพธ์หลัง JOIN
19. เก็บ Query เป็น Python Function
20. แบบฝึกหัดท้ายบท

## 1. แนวคิดการใช้ SQL กับงานวิเคราะห์ข้อมูล

SQL ย่อมาจาก **Structured Query Language**

เป็นภาษาสำหรับทำงานกับข้อมูลในรูปแบบตาราง

ในงานวิเคราะห์ เรามักใช้ SQL เพื่อตอบคำถาม เช่น

- ตารางมีข้อมูลกี่รายการ
- จังหวัดใดมีจำนวนเกษตรกรมากที่สุด
- แต่ละพื้นที่มีรายได้หรือหนี้สินเฉลี่ยเท่าใด
- มีครัวเรือนที่ไม่ซ้ำกี่ครัวเรือน
- พื้นที่ใดมีข้อมูลอยู่ในทั้งสองแหล่ง
- พื้นที่ใดพบเฉพาะในแหล่งข้อมูลหนึ่ง
- พื้นที่ใดมีความแตกต่างระหว่างจำนวนข้อมูลสองระบบมากที่สุด

ในบทนี้ SQL จะถูกใช้ผ่าน DuckDB เพื่อ Query pandas DataFrame โดยตรง

### เปรียบเทียบ pandas กับ SQL

งานเดียวกันสามารถทำได้ทั้ง pandas และ SQL

| งาน | pandas | SQL |
|---|---|---|
| เลือกคอลัมน์ | `df[["a", "b"]]` | `SELECT a, b` |
| กรองข้อมูล | `df.loc[condition]` | `WHERE condition` |
| เรียงข้อมูล | `.sort_values()` | `ORDER BY` |
| สรุปตามกลุ่ม | `.groupby()` | `GROUP BY` |
| เชื่อมตาราง | `.merge()` | `JOIN` |
| เติมค่าว่าง | `.fillna()` | `COALESCE()` |

การเลือกใช้ขึ้นอยู่กับ

- ความถนัดของทีม
- ความซับซ้อนของ Query
- ระบบที่นำไปใช้งาน
- ความต้องการนำ Query ไปใช้กับฐานข้อมูลจริง

## 2. เตรียม Library และข้อมูลตัวอย่าง

Library ที่ใช้ในบทนี้ ได้แก่

- `pandas` สำหรับอ่านและเตรียมข้อมูล
- `duckdb` สำหรับประมวลผล SQL
- `pathlib.Path` สำหรับจัดการตำแหน่งไฟล์
```

In [1]:
%pip install duckdb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import duckdb
import pandas as pd

บทนี้ใช้ข้อมูล 2 ชุด ได้แก่

1. ข้อมูลเกษตรกรเปราะบางจากไฟล์ Excel
2. ข้อมูล Logbook จากไฟล์ CSV

กำหนด Path โดยสมมติว่าไฟล์อยู่ใน Folder เดียวกับ Notebook

In [3]:
farmer_path = Path(
    "moac_opsmoac_fragile_farmer.xlsx"
)

mso_path = Path(
    "msdhs_ops_mso_logbook.csv"
)

In [6]:
mso_usecols = [
    "วันที่แก้ไขข้อมูลล่าสุด",
    "รหัสครัวเรือน",
    "รหัสประจำบ้าน",
    "วันที่สร้างครัวเรือน",
    "ตำบล/แขวง",
    "เขต/อำเภอ/เทศบาล",
    "จังหวัด",
    "อายุ",
    "เพศ",
    "ระดับการศึกษา",
    "อาชีพหลัก",
    "ประเภทกลุ่มเป้าหมาย",
    "รหัส cm",
    "หน่วยงานของ cm",
]

mso_dtype = {
    "รหัสครัวเรือน": "string",
    "รหัสประจำบ้าน": "string",
    "รหัส cm": "string",
}


In [7]:
mso_df = pd.read_csv(
    mso_path,
    encoding="utf-8-sig",
    usecols=mso_usecols,
    dtype=mso_dtype,
    parse_dates=[
        "วันที่แก้ไขข้อมูลล่าสุด",
        "วันที่สร้างครัวเรือน",
    ],
)

mso_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'msdhs_ops_mso_logbook.csv'

อย่างไรก็ตาม ในการใช้งานจริง ไฟล์ข้อมูลอาจไม่ได้อยู่ใน folder เดียวกับ notebook เสมอไป ดังนั้นก่อนอ่านไฟล์ ควรตรวจสอบตำแหน่งปัจจุบันของ notebook และตรวจสอบว่าไฟล์ข้อมูลอยู่ที่ path ใด หากระบุ path ไม่ถูกต้อง เช่น ใช้แค่ชื่อไฟล์โดยตรง แต่ไฟล์อยู่คนละ directory จะเกิด error เช่น

```text 
FileNotFoundError: [Errno 2] No such file or directory
```

ดังนั้นขั้นตอนที่ดีคือ
1. ตรวจสอบ current working directory
2. กำหนด folder ที่เก็บข้อมูล
3. ประกอบ path ของไฟล์ด้วย pathlib.Path
4. ตรวจสอบว่าไฟล์มีอยู่จริงก่อนอ่าน

ตรวจสอบว่า notebook กำลังทำงานอยู่ที่ directory ใด

In [8]:
current_dir = Path.cwd() 
current_dir

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day6')

กำหนด directory ที่เก็บข้อมูล

ตัวอย่างนี้ไฟล์ข้อมูลอยู่ไม่ได้อยู่ใน folder เดียวกับ notebook

โครงสร้าง folder ตัวอย่าง:

```text
course/
├── day5/
│   └── basic_sql.ipynb
└── day3/
    ├── moac_opsmoac_fragile_farmer.xlsx
    └── msdhs_ops_mso_logbook.csv
```

ถ้า notebook อยู่ใน folder notebooks และข้อมูลอยู่ใน folder data ที่อยู่ข้างนอก notebook folder
จะต้องใช้ path `/workspaces/MSDHS_OJT/py-jupyter_docker/course`

In [9]:
data_dir = Path("/workspaces/MSDHS_OJT/py-jupyter_docker/course") 
data_dir

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course')

กำหนด path ของไฟล์ข้อมูลแต่ละไฟล์

In [10]:
farmer_path = data_dir / "day3/moac_opsmoac_fragile_farmer.xlsx" 
mso_path = data_dir / "day3/msdhs_ops_mso_logbook.csv" 

In [11]:
farmer_path

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3/moac_opsmoac_fragile_farmer.xlsx')

In [12]:
mso_path

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3/msdhs_ops_mso_logbook.csv')

ตรวจสอบว่าไฟล์มีอยู่จริงหรือไม่ก่อนอ่านข้อมูล 
ถ้าผลลัพธ์เป็น `False` แปลว่า path ยังไม่ถูกต้อง ต้องกลับไปตรวจสอบว่า 
- notebook อยู่ที่ directory ใด 
- ไฟล์ข้อมูลอยู่ที่ directory ใด 
- ชื่อไฟล์สะกดถูกต้องหรือไม่ 
- นามสกุลไฟล์ถูกต้องหรือไม่ เช่น `.xlsx`, `.csv`

In [13]:
print("Farmer file exists:", farmer_path.exists())

Farmer file exists: True


In [14]:
print("MSO file exists:", mso_path.exists())

MSO file exists: True


ถ้าต้องการดูไฟล์ทั้งหมดใน folder ข้อมูล สามารถใช้คำสั่งนี้เพื่อตรวจสอบชื่อไฟล์

In [15]:
list(data_dir.iterdir())

[PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3'),
 PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day6'),
 PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day2')]

หลังจากตรวจสอบแล้วว่า path ถูกต้อง จึงอ่านไฟล์เข้ามาเป็น DataFrame

In [16]:
mso_df = pd.read_csv(
    mso_path,
    encoding="utf-8-sig",
    usecols=mso_usecols,
    dtype=mso_dtype,
    parse_dates=[
        "วันที่แก้ไขข้อมูลล่าสุด",
        "วันที่สร้างครัวเรือน",
    ],
)

mso_df.head()

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,ตำบล/แขวง,เขต/อำเภอ/เทศบาล,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


In [17]:
target_sheets = [
    "v_cpd_fragile",
    "v_dld_fragile",
    "v_doae_fragile",
]

farmer_usecols = [
    "department_code",
    "department",
    "pid",
    "province_code",
    "province",
    "amphur",
    "tambon",
    "is_farmer",
    "farmer_type",
    "main_occupation",
    "income_in",
    "income_out",
    "debts_in",
    "debts_out",
    "updated_at",
]

farmer_dtype = {
    "department_code": "string",
    "pid": "string",
    "province_code": "string",
}

ตรวจสอบว่า Worksheet ที่ต้องการมีอยู่ใน Workbook

In [18]:
excel_file = pd.ExcelFile(
    farmer_path
)

excel_file.sheet_names

['v_cpd_fragile',
 'v_dld_fragile',
 'v_doae_fragile',
 'v_dof_fragile',
 'v_raot_fragile']

In [19]:
farmer_df_list = []

for sheet_name in target_sheets:
    if (
        sheet_name
        not in excel_file.sheet_names
    ):
        print(
            f"Sheet not found: {sheet_name}"
        )
        continue

    temp_df = pd.read_excel(
        excel_file,
        sheet_name=sheet_name,
        usecols=farmer_usecols,
        dtype=farmer_dtype,
    )

    temp_df["source_sheet"] = (
        sheet_name
    )

    farmer_df_list.append(
        temp_df
    )

In [20]:
farmer_raw_df = pd.concat(
    farmer_df_list,
    ignore_index=True,
)

farmer_raw_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


## 4. ตรวจสอบข้อมูลหลังนำเข้า

ก่อน Query ควรตรวจสอบอย่างน้อย

- จำนวนแถวและคอลัมน์
- ตัวอย่างข้อมูล
- ชื่อคอลัมน์
- ชนิดข้อมูล
- ค่าว่างใน Key ที่จะใช้เชื่อม

In [21]:
print(
    "Farmer shape:",
    farmer_raw_df.shape,
)

farmer_raw_df.head()

Farmer shape: (95, 16)


,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


In [22]:
farmer_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   department_code  95 non-null     string 
 1   department       95 non-null     str    
 2   pid              95 non-null     string 
 3   province_code    67 non-null     string 
 4   province         10 non-null     object 
 5   amphur           10 non-null     object 
 6   tambon           10 non-null     object 
 7   is_farmer        95 non-null     int64  
 8   farmer_type      95 non-null     str    
 9   main_occupation  21 non-null     object 
 10  income_in        64 non-null     float64
 11  income_out       64 non-null     float64
 12  debts_in         64 non-null     float64
 13  debts_out        64 non-null     float64
 14  updated_at       80 non-null     float64
 15  source_sheet     95 non-null     str    
dtypes: float64(5), int64(1), object(4), str(3), string(3)
memory usage: 12.0+ K

In [19]:
farmer_raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   department_code  95 non-null     str    
 1   department       95 non-null     str    
 2   pid              95 non-null     string 
 3   province_code    67 non-null     string 
 4   province         10 non-null     object 
 5   amphur           10 non-null     object 
 6   tambon           10 non-null     object 
 7   is_farmer        95 non-null     int64  
 8   farmer_type      95 non-null     str    
 9   main_occupation  21 non-null     object 
 10  income_in        64 non-null     float64
 11  income_out       64 non-null     float64
 12  debts_in         64 non-null     float64
 13  debts_out        64 non-null     float64
 14  updated_at       80 non-null     float64
 15  source_sheet     95 non-null     str    
dtypes: float64(5), int64(1), object(4), str(4), string(2)
memory usage: 12.0+ K

In [23]:
print(
    "MSO shape:",
    mso_df.shape,
)

mso_df.head()

MSO shape: (100, 14)


,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,ตำบล/แขวง,เขต/อำเภอ/เทศบาล,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


In [24]:
mso_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   วันที่แก้ไขข้อมูลล่าสุด  100 non-null    datetime64[us]
 1   รหัสครัวเรือน            100 non-null    string        
 2   รหัสประจำบ้าน            24 non-null     string        
 3   วันที่สร้างครัวเรือน     100 non-null    datetime64[us]
 4   อายุ                     100 non-null    int64         
 5   เพศ                      77 non-null     str           
 6   ตำบล/แขวง                100 non-null    str           
 7   เขต/อำเภอ/เทศบาล         100 non-null    str           
 8   จังหวัด                  100 non-null    str           
 9   ระดับการศึกษา            77 non-null     str           
 10  อาชีพหลัก                39 non-null     str           
 11  ประเภทกลุ่มเป้าหมาย      100 non-null    str           
 12  รหัส cm                  100 non-null    string 

## 5. เตรียม DataFrame สำหรับ Query

เพื่อให้เขียน SQL ได้ง่ายขึ้น จะดำเนินการดังนี้

1. สร้างสำเนาของข้อมูลเกษตรกร
2. เปลี่ยนชื่อคอลัมน์ MSO เป็นภาษาอังกฤษ
3. ทำชื่อ Key ของทั้งสองชุดให้ตรงกัน
4. ทำความสะอาด Key
5. สร้างตัวแปรสำหรับการวิเคราะห์

In [25]:
farmer_df = (
    farmer_raw_df.copy()
)

mso_df = mso_df.rename(
    columns={
        "จังหวัด": "province",
        "เขต/อำเภอ/เทศบาล": (
            "amphur"
        ),
        "ตำบล/แขวง": "tambon",
        "วันที่แก้ไขข้อมูลล่าสุด": (
            "last_updated_at"
        ),
        "วันที่สร้างครัวเรือน": (
            "household_created_at"
        ),
        "รหัสครัวเรือน": (
            "household_id"
        ),
        "รหัสประจำบ้าน": (
            "house_code"
        ),
        "อายุ": "age",
        "เพศ": "gender",
        "ระดับการศึกษา": (
            "education_level"
        ),
        "อาชีพหลัก": (
            "main_occupation"
        ),
        "ประเภทกลุ่มเป้าหมาย": (
            "target_group"
        ),
        "รหัส cm": "cm_id",
        "หน่วยงานของ cm": (
            "cm_department"
        ),
    }
)

In [26]:
farmer_df.columns.tolist()

['department_code',
 'department',
 'pid',
 'province_code',
 'province',
 'amphur',
 'tambon',
 'is_farmer',
 'farmer_type',
 'main_occupation',
 'income_in',
 'income_out',
 'debts_in',
 'debts_out',
 'updated_at',
 'source_sheet']

In [27]:
mso_df.columns.tolist()

['last_updated_at',
 'household_id',
 'house_code',
 'household_created_at',
 'age',
 'gender',
 'tambon',
 'amphur',
 'province',
 'education_level',
 'main_occupation',
 'target_group',
 'cm_id',
 'cm_department']

### ทำความสะอาด Key สำหรับ JOIN

บทนี้ใช้ชื่อพื้นที่เป็น Composite Key ได้แก่

- `province`
- `amphur`
- `tambon`

Key ที่เป็นข้อความอาจมีปัญหา เช่น

- ช่องว่างหัวท้าย
- ค่าว่าง
- การสะกดต่างกัน
- คำนำหน้าพื้นที่ต่างกัน
- รูปแบบชื่อไม่สอดคล้องกัน

ขั้นแรกจะปรับเป็น String และตัดช่องว่างหัวท้าย

In [28]:
area_columns = [
    "province",
    "amphur",
    "tambon",
]

for column in area_columns:
    farmer_df[column] = (
        farmer_df[column]
        .astype("string")
        .str.strip()
    )

    mso_df[column] = (
        mso_df[column]
        .astype("string")
        .str.strip()
    )

ตรวจสอบค่าว่างใน Key

In [29]:
farmer_df[
    area_columns
].isna().sum()

province    85
amphur      85
tambon      85
dtype: int64

In [30]:
mso_df[
    area_columns
].isna().sum()

province    0
amphur      0
tambon      0
dtype: int64

ตรวจสอบจำนวนพื้นที่ไม่ซ้ำในแต่ละแหล่ง

In [31]:
farmer_area_count = (
    farmer_df[
        area_columns
    ]
    .drop_duplicates()
    .shape[0]
)

farmer_area_count

11

In [32]:
mso_area_count = (
    mso_df[
        area_columns
    ]
    .drop_duplicates()
    .shape[0]
)

mso_area_count

99

### สร้างคอลัมน์สำหรับการวิเคราะห์

ข้อมูลเกษตรกรมีรายได้และหนี้แยกเป็น

- ในภาคเกษตร
- นอกภาคเกษตร

จึงสร้าง

- `total_income`
- `total_debt`

ก่อน Query

In [33]:
income_columns = [
    "income_in",
    "income_out",
]

debt_columns = [
    "debts_in",
    "debts_out",
]

for column in (
    income_columns
    + debt_columns
):
    farmer_df[column] = (
        pd.to_numeric(
            farmer_df[column],
            errors="coerce",
        )
    )

In [34]:
farmer_df["total_income"] = (
    farmer_df[
        income_columns
    ]
    .sum(
        axis=1,
        min_count=1,
    )
)

farmer_df["total_debt"] = (
    farmer_df[
        debt_columns
    ]
    .sum(
        axis=1,
        min_count=1,
    )
)

การใช้ `min_count=1` ทำให้ผลรวมเป็นค่าว่าง หากทั้งสองคอลัมน์ต้นทางเป็นค่าว่าง

วิธีนี้ต่างจากการแปลงค่าว่างทั้งหมดเป็นศูนย์โดยอัตโนมัติ

In [35]:
mso_df["age"] = (
    pd.to_numeric(
        mso_df["age"],
        errors="coerce",
    )
)

In [36]:
farmer_df[
    [
        "pid",
        "province",
        "amphur",
        "tambon",
        "total_income",
        "total_debt",
    ]
].head()

,pid,province,amphur,tambon,total_income,total_debt
0,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,<NA>,<NA>,NaN,NaN
1,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,<NA>,<NA>,NaN,NaN
2,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,<NA>,<NA>,NaN,NaN
3,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,<NA>,<NA>,NaN,NaN
4,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,<NA>,<NA>,NaN,NaN


In [38]:
mso_df[
    [
        "household_id",
        "province",
        "amphur",
        "tambon",
        "age",
        "target_group",
    ]
].head()

,household_id,province,amphur,tambon,age,target_group
0,648ff7d6b64d06b6b0dc7f02,อุดรธานี,นายูง,โนนทอง,3,เด็กเล็ก
1,62d3f7a6d6f101550054198b,ราชบุรี,ดำเนินสะดวก,ท่านัด,55,วัยผู้ใหญ่/วัยแรงงาน
2,62a05c9df5674ecdd3805787,หนองคาย,ท่าบ่อ,นาข่า,33,วัยผู้ใหญ่/วัยแรงงาน
3,62908e378fa67bf5ad7d5555,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,5,เด็กเล็ก
4,6319b26ca3a37508384e36fc,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,3,เด็กเล็ก


## 6. สร้าง DuckDB Connection

DuckDB ทำหน้าที่เป็น SQL Engine

คำสั่งต่อไปนี้สร้าง Connection แบบชั่วคราวในหน่วยความจำ

In [39]:
con = duckdb.connect()

## 7. Register DataFrame เป็นตาราง

ก่อนใช้ SQL ต้อง Register DataFrame ให้ DuckDB รู้จัก

ในบทนี้ใช้ชื่อตารางว่า

- `farmer`
- `mso`

In [40]:
con.register(
    "farmer",
    farmer_df,
)

con.register(
    "mso",
    mso_df,
)

ทดลอง Query ตารางละ 5 แถว

In [41]:
con.sql(
    """
    SELECT *
    FROM farmer
    LIMIT 5
    """
).df()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet,total_income,total_debt
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,NaN,NaN
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,NaN,NaN
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,NaN,NaN
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,NaN,NaN
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,None,None,None,None,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile,NaN,NaN


In [42]:
con.sql(
    """
    SELECT *
    FROM mso
    LIMIT 5
    """
).df()

,last_updated_at,household_id,house_code,household_created_at,age,gender,tambon,amphur,province,education_level,main_occupation,target_group,cm_id,cm_department
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,NaN,2023-06-19 13:38:14.881,3,หญิง,โนนทอง,นายูง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ท่านัด,ดำเนินสะดวก,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,NaN,2022-08-06 15:23:57.850,33,หญิง,นาข่า,ท่าบ่อ,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,NaN,2022-05-27 15:39:19.197,5,หญิง,โดมประดิษฐ์,น้ำยืน,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,NaN,2022-08-09 16:14:20.652,3,ชาย,หนองหญ้าขาว,สีคิ้ว,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


`con.sql(...).df()` ทำงานดังนี้

1. ส่ง SQL ให้ DuckDB ประมวลผล
2. รับผลลัพธ์จาก Query
3. แปลงผลลัพธ์กลับเป็น pandas DataFrame

## 8. เลือกข้อมูลด้วย `SELECT`

`SELECT` ใช้กำหนดคอลัมน์ที่ต้องการจากตาราง

รูปแบบพื้นฐานคือ

```sql
SELECT column_1, column_2
FROM table_name
```

ไม่ควรใช้ `SELECT *` โดยไม่จำเป็นใน Query สำหรับใช้งานจริง เพราะทำให้

- อ่านผลลัพธ์ยาก
- ดึงคอลัมน์ที่ไม่จำเป็น
- Schema ของผลลัพธ์ไม่ชัดเจน

In [43]:
con.sql(
    """
    SELECT
        department,
        source_sheet,
        province,
        amphur,
        tambon,
        farmer_type,
        main_occupation
    FROM farmer
    LIMIT 10
    """
).df()

,department,source_sheet,province,amphur,tambon,farmer_type,main_occupation
0,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
1,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
2,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
3,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
4,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
5,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
6,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
7,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
8,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None
9,กรมส่งเสริมสหกรณ์,v_cpd_fragile,None,None,None,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,None


In [44]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        age,
        gender,
        education_level,
        target_group,
        cm_department
    FROM mso
    LIMIT 10
    """
).df()

,province,amphur,tambon,age,gender,education_level,target_group,cm_department
0,อุดรธานี,นายูง,โนนทอง,3,หญิง,ไม่ได้เรียนหนังสือ,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,ราชบุรี,ดำเนินสะดวก,ท่านัด,55,หญิง,ไม่ได้เรียนหนังสือ,วัยผู้ใหญ่/วัยแรงงาน,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,หนองคาย,ท่าบ่อ,นาข่า,33,หญิง,มัธยมศึกษาตอนต้น,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,5,หญิง,ไม่ได้เรียนหนังสือ,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,3,ชาย,ไม่ได้เรียนหนังสือ,เด็กเล็ก,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
5,ประจวบคีรีขันธ์,เมืองประจวบคีรีขันธ์,อ่าวน้อย,39,ชาย,มัธยมศึกษาตอนต้น,วัยผู้ใหญ่/วัยแรงงาน,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
6,อุดรธานี,เทศบาลนครอุดรธานี,หมากแข้ง,4,หญิง,ไม่ได้เรียนหนังสือ,เด็กเล็ก,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
7,นครศรีธรรมราช,ปากพนัง,ปากพนังฝั่งตะวันออก,68,NaN,NaN,ผู้สูงอายุ,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.นครศรีธรรมราช
8,สระแก้ว,วัฒนานคร,วัฒนานคร,90,หญิง,ประถมศึกษา,ผู้สูงอายุ,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
9,บุรีรัมย์,หนองหงส์,ไทยสามัคคี,6,หญิง,ไม่ได้เรียนหนังสือ,เด็ก,บ้านพักเด็กและครอบครัวจังหวัดบุรีรัมย์


### ตั้งชื่อผลลัพธ์ด้วย Alias

ใช้ `AS` เพื่อกำหนดชื่อคอลัมน์ในผลลัพธ์

In [45]:
con.sql(
    """
    SELECT
        province AS province_name,
        total_income AS income,
        total_debt AS debt
    FROM farmer
    LIMIT 10
    """
).df()

,province_name,income,debt
0,None,NaN,NaN
1,None,NaN,NaN
2,None,NaN,NaN
3,None,NaN,NaN
4,None,NaN,NaN
5,None,NaN,NaN
6,None,NaN,NaN
7,None,NaN,NaN
8,None,NaN,NaN
9,None,NaN,NaN


## 9. กรองข้อมูลด้วย `WHERE`

`WHERE` ใช้เลือกเฉพาะแถวที่เข้าเงื่อนไข

ตัวอย่าง Operator ที่ใช้บ่อย ได้แก่

| Operator | ความหมาย |
|---|---|
| `=` | เท่ากับ |
| `<>` หรือ `!=` | ไม่เท่ากับ |
| `>` | มากกว่า |
| `<` | น้อยกว่า |
| `>=` | มากกว่าหรือเท่ากับ |
| `<=` | น้อยกว่าหรือเท่ากับ |
| `AND` | ทุกเงื่อนไขเป็นจริง |
| `OR` | อย่างน้อยหนึ่งเงื่อนไขเป็นจริง |
| `IN` | อยู่ในรายการ |
| `BETWEEN` | อยู่ในช่วง |

In [46]:
con.sql(
    """
    SELECT
        department,
        province,
        amphur,
        tambon,
        total_income,
        total_debt
    FROM farmer
    WHERE province = 'อุดรธานี'
    LIMIT 10
    """
).df()

,department,province,amphur,tambon,total_income,total_debt
0,กรมปศุสัตว์,อุดรธานี,เพ็ญ,นาพู่,NaN,NaN


### ใช้หลายเงื่อนไข

In [47]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        total_income,
        total_debt
    FROM farmer
    WHERE
        province = 'อุดรธานี'
        AND total_debt > total_income
    LIMIT 10
    """
).df()

,province,amphur,tambon,total_income,total_debt


### ใช้ `IN`

In [48]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        total_income
    FROM farmer
    WHERE province IN (
        'อุดรธานี',
        'ขอนแก่น',
        'เชียงใหม่'
    )
    LIMIT 10
    """
).df()

,province,amphur,tambon,total_income
0,อุดรธานี,เพ็ญ,นาพู่,NaN
1,ขอนแก่น,บ้านไผ่,ป่าปอ,NaN


### ตรวจสอบ `NULL`

ใน SQL ไม่ควรเขียน

```sql
column = NULL
```

ให้ใช้

```sql
column IS NULL
column IS NOT NULL
```

In [49]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        total_income
    FROM farmer
    WHERE total_income IS NOT NULL
    LIMIT 10
    """
).df()


,province,amphur,tambon,total_income
0,None,None,None,0.0
1,None,None,None,35000.0
2,None,None,None,110000.0
3,None,None,None,100000.0
4,None,None,None,80000.0
5,None,None,None,250000.0
6,None,None,None,120000.0
7,None,None,None,0.0
8,None,None,None,343200.0
9,None,None,None,0.0


## 10. เรียงและจำกัดผลลัพธ์

`ORDER BY` ใช้เรียงผลลัพธ์

- `ASC` เรียงจากน้อยไปมาก
- `DESC` เรียงจากมากไปน้อย

`LIMIT` ใช้จำกัดจำนวนแถวของผลลัพธ์

In [50]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        total_income,
        total_debt
    FROM farmer
    WHERE province IS NOT NULL
    ORDER BY total_income ASC
    LIMIT 10
    """
).df()

,province,amphur,tambon,total_income,total_debt
0,บุรีรัมย์,นางรอง,ก้านเหลือง,NaN,NaN
1,อุบลราชธานี,ตระการพืชผล,หนองเต่า,NaN,NaN
2,อุดรธานี,เพ็ญ,นาพู่,NaN,NaN
3,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,NaN,NaN
4,แพร่,ร้องกวาง,แม่ยางร้อง,NaN,NaN
5,อุบลราชธานี,ตระการพืชผล,สะพือ,NaN,NaN
6,ขอนแก่น,บ้านไผ่,ป่าปอ,NaN,NaN
7,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,NaN,NaN
8,สุรินทร์,รัตนบุรี,เบิด,NaN,NaN
9,พะเยา,ภูซาง,ทุ่งกล้วย,NaN,NaN


In [52]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        total_income,
        total_debt
    FROM farmer
    WHERE province IS NOT NULL
    ORDER BY total_debt DESC
    LIMIT 10
    """
).df()

,province,amphur,tambon,total_income,total_debt
0,บุรีรัมย์,นางรอง,ก้านเหลือง,NaN,NaN
1,อุบลราชธานี,ตระการพืชผล,หนองเต่า,NaN,NaN
2,อุดรธานี,เพ็ญ,นาพู่,NaN,NaN
3,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,NaN,NaN
4,แพร่,ร้องกวาง,แม่ยางร้อง,NaN,NaN
5,อุบลราชธานี,ตระการพืชผล,สะพือ,NaN,NaN
6,ขอนแก่น,บ้านไผ่,ป่าปอ,NaN,NaN
7,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,NaN,NaN
8,สุรินทร์,รัตนบุรี,เบิด,NaN,NaN
9,พะเยา,ภูซาง,ทุ่งกล้วย,NaN,NaN


สามารถเรียงหลายคอลัมน์ได้

ตัวอย่างต่อไปนี้เรียงจังหวัดตามตัวอักษร และเรียงหนี้จากมากไปน้อยภายในจังหวัด

In [53]:
con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        total_debt
    FROM farmer
    WHERE province IS NOT NULL
    ORDER BY
        province ASC,
        total_debt DESC
    LIMIT 20
    """
).df()

,province,amphur,tambon,total_debt
0,ขอนแก่น,บ้านไผ่,ป่าปอ,NaN
1,บุรีรัมย์,นางรอง,ก้านเหลือง,NaN
2,พะเยา,ภูซาง,ทุ่งกล้วย,NaN
3,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,NaN
4,สุรินทร์,รัตนบุรี,เบิด,NaN
5,อุดรธานี,เพ็ญ,นาพู่,NaN
6,อุบลราชธานี,ตระการพืชผล,หนองเต่า,NaN
7,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,NaN
8,อุบลราชธานี,ตระการพืชผล,สะพือ,NaN
9,แพร่,ร้องกวาง,แม่ยางร้อง,NaN


## 11. สร้างเงื่อนไขด้วย `CASE WHEN`

`CASE WHEN` ใช้สร้างค่าตามเงื่อนไข คล้าย `if-elif-else`

รูปแบบคือ

```sql
CASE
    WHEN condition_1 THEN result_1
    WHEN condition_2 THEN result_2
    ELSE default_result
END
```

In [54]:
con.sql(
    """
    SELECT
        province,
        total_income,
        total_debt,
        CASE
            WHEN total_debt > total_income
                THEN 'debt_above_income'
            WHEN total_debt = 0
                THEN 'no_debt'
            ELSE 'debt_within_income'
        END AS debt_status
    FROM farmer
    LIMIT 20
    """
).df()

,province,total_income,total_debt,debt_status
0,NaN,NaN,NaN,debt_within_income
1,NaN,NaN,NaN,debt_within_income
2,NaN,NaN,NaN,debt_within_income
3,NaN,NaN,NaN,debt_within_income
4,NaN,NaN,NaN,debt_within_income
5,NaN,NaN,NaN,debt_within_income
6,NaN,NaN,NaN,debt_within_income
7,NaN,NaN,NaN,debt_within_income
8,NaN,NaN,NaN,debt_within_income
9,NaN,NaN,NaN,debt_within_income


## 12. Aggregate Function

Aggregate Function ใช้สรุปข้อมูลหลายแถวเป็นค่าหนึ่งค่า

| Function | ความหมาย |
|---|---|
| `COUNT(*)` | นับจำนวนแถว |
| `COUNT(column)` | นับค่าที่ไม่เป็น `NULL` |
| `COUNT(DISTINCT column)` | นับค่าที่ไม่ซ้ำและไม่เป็น `NULL` |
| `SUM(column)` | ผลรวม |
| `AVG(column)` | ค่าเฉลี่ย |
| `MIN(column)` | ค่าน้อยที่สุด |
| `MAX(column)` | ค่ามากที่สุด |

ต้องระวังความแตกต่างระหว่างจำนวนแถวกับจำนวนหน่วยข้อมูลที่ไม่ซ้ำ

In [55]:
con.sql(
    """
    SELECT
        COUNT(*) AS farmer_records,
        COUNT(pid) AS non_null_pid_records,
        COUNT(DISTINCT pid)
            AS unique_farmers,
        AVG(total_income)
            AS avg_total_income,
        AVG(total_debt)
            AS avg_total_debt,
        SUM(total_income)
            AS sum_total_income,
        SUM(total_debt)
            AS sum_total_debt
    FROM farmer
    """
).df()

,farmer_records,non_null_pid_records,unique_farmers,avg_total_income,avg_total_debt,sum_total_income,sum_total_debt
0,95,95,95,88165.625,39765.625,5642600.0,2545000.0


`COUNT(*)` และ `COUNT(DISTINCT pid)` อาจให้ค่าต่างกัน เพราะ

- บุคคลหนึ่งคนอาจมีหลายแถว
- `pid` อาจซ้ำ
- `pid` อาจเป็น `NULL`

## 13. จัดกลุ่มด้วย `GROUP BY`

`GROUP BY` ใช้สรุปข้อมูลแยกตามหมวดหมู่หรือพื้นที่

ทุกคอลัมน์ใน `SELECT` ที่ไม่ได้อยู่ใน Aggregate Function ต้องอยู่ใน `GROUP BY`

In [56]:
con.sql(
    """
    SELECT
        province,
        department,
        COUNT(*) AS farmer_records,
        COUNT(DISTINCT pid)
            AS unique_farmers
    FROM farmer
    GROUP BY
        province,
        department
    ORDER BY farmer_records DESC
    LIMIT 10
    """
).df()

,province,department,farmer_records,unique_farmers
0,NaN,กรมส่งเสริมการเกษตร,70,70
1,NaN,กรมส่งเสริมสหกรณ์,15,15
2,อุบลราชธานี,กรมปศุสัตว์,3,3
3,อุดรธานี,กรมปศุสัตว์,1,1
4,สุรินทร์,กรมปศุสัตว์,1,1
5,ขอนแก่น,กรมปศุสัตว์,1,1
6,พะเยา,กรมปศุสัตว์,1,1
7,ศรีสะเกษ,กรมปศุสัตว์,1,1
8,แพร่,กรมปศุสัตว์,1,1
9,บุรีรัมย์,กรมปศุสัตว์,1,1


In [57]:
con.sql(
    """
    SELECT
        province,
        target_group,
        COUNT(*) AS mso_records,
        COUNT(DISTINCT household_id)
            AS unique_households
    FROM mso
    GROUP BY
        province,
        target_group
    ORDER BY mso_records DESC
    LIMIT 10
    """
).df()

,province,target_group,mso_records,unique_households
0,หนองคาย,วัยผู้ใหญ่/วัยแรงงาน,4,4
1,เชียงใหม่,วัยผู้ใหญ่/วัยแรงงาน,3,3
2,ขอนแก่น,ผู้สูงอายุ,3,3
3,นครศรีธรรมราช,วัยผู้ใหญ่/วัยแรงงาน,3,3
4,อุทัยธานี,วัยผู้ใหญ่/วัยแรงงาน,3,3
5,นครศรีธรรมราช,ผู้สูงอายุ,3,3
6,อุดรธานี,เด็กเล็ก,3,3
7,อุดรธานี,วัยผู้ใหญ่/วัยแรงงาน,3,3
8,นครราชสีมา,ผู้สูงอายุ,3,3
9,ราชบุรี,วัยผู้ใหญ่/วัยแรงงาน,2,2


### ใช้ `HAVING` กรองผลหลังสรุป

- `WHERE` กรองแถวก่อน `GROUP BY`
- `HAVING` กรองกลุ่มหลัง `GROUP BY`

In [58]:
con.sql(
    """
    SELECT
        province,
        COUNT(*) AS farmer_records,
        COUNT(DISTINCT pid)
            AS unique_farmers
    FROM farmer
    WHERE province IS NOT NULL
    GROUP BY province
    HAVING COUNT(*) >= 10
    ORDER BY farmer_records DESC
    """
).df()

,province,farmer_records,unique_farmers


## 14. สรุปข้อมูลระดับจังหวัด

ตัวอย่างสรุปข้อมูลเกษตรกร

In [59]:
con.sql(
    """
    SELECT
        province,
        COUNT(*) AS farmer_records,
        COUNT(DISTINCT pid)
            AS unique_farmers,
        AVG(total_income)
            AS avg_total_income,
        AVG(total_debt)
            AS avg_total_debt,
        SUM(total_income)
            AS sum_total_income,
        SUM(total_debt)
            AS sum_total_debt
    FROM farmer
    GROUP BY province
    ORDER BY farmer_records DESC
    LIMIT 10
    """
).df()

,province,farmer_records,unique_farmers,avg_total_income,avg_total_debt,sum_total_income,sum_total_debt
0,NaN,85,85,88165.625,39765.625,5642600.0,2545000.0
1,อุบลราชธานี,3,3,NaN,NaN,NaN,NaN
2,อุดรธานี,1,1,NaN,NaN,NaN,NaN
3,ศรีสะเกษ,1,1,NaN,NaN,NaN,NaN
4,ขอนแก่น,1,1,NaN,NaN,NaN,NaN
5,บุรีรัมย์,1,1,NaN,NaN,NaN,NaN
6,แพร่,1,1,NaN,NaN,NaN,NaN
7,พะเยา,1,1,NaN,NaN,NaN,NaN
8,สุรินทร์,1,1,NaN,NaN,NaN,NaN


ตัวอย่างสรุปข้อมูล MSO

In [60]:
con.sql(
    """
    SELECT
        province,
        COUNT(*) AS mso_records,
        COUNT(DISTINCT household_id)
            AS unique_households,
        AVG(age) AS avg_age,
        MIN(age) AS min_age,
        MAX(age) AS max_age
    FROM mso
    GROUP BY province
    ORDER BY mso_records DESC
    LIMIT 10
    """
).df()

,province,mso_records,unique_households,avg_age,min_age,max_age
0,นครศรีธรรมราช,8,8,39.625000,5,68
1,อุดรธานี,7,7,27.142857,3,62
2,เชียงใหม่,5,5,30.600000,3,54
3,หนองคาย,4,4,36.500000,31,44
4,นครราชสีมา,4,4,61.000000,3,103
5,บุรีรัมย์,4,4,30.000000,6,79
6,อุทัยธานี,4,4,53.000000,38,65
7,ขอนแก่น,4,4,70.500000,58,83
8,นครนายก,4,4,36.750000,13,70
9,นครพนม,3,3,38.666667,17,54


## 15. ทำความเข้าใจ Grain ก่อน JOIN

**Grain** หมายถึงระดับรายละเอียดของหนึ่งแถว

ตัวอย่างเช่น

- หนึ่งแถวต่อบุคคล
- หนึ่งแถวต่อครัวเรือน
- หนึ่งแถวต่อเหตุการณ์
- หนึ่งแถวต่อจังหวัด
- หนึ่งแถวต่อจังหวัด–อำเภอ–ตำบล

ก่อน JOIN ต้องรู้ว่าแต่ละตารางมี Grain อะไร

ข้อมูลในบทนี้อาจมี Grain ต่างกัน

- `farmer` อาจมีหนึ่งแถวต่อบุคคลหรือรายการเกษตรกร
- `mso` อาจมีหนึ่งแถวต่อสมาชิกครัวเรือนหรือรายการ Logbook

หาก JOIN แบบ Row-level ด้วยพื้นที่ทันที อาจเกิด **Many-to-many Join**

ตัวอย่าง:

- Farmer มี 10 แถวในพื้นที่หนึ่ง
- MSO มี 20 แถวในพื้นที่เดียวกัน
- JOIN ตรง ๆ อาจได้ 200 แถว

ดังนั้น ในบทนี้จะสรุปทั้งสองตารางให้อยู่ใน Grain เดียวกันก่อน JOIN

Grain ที่ใช้คือ

```text
province + amphur + tambon
```

หนึ่งแถวของ Summary จึงแทนหนึ่งพื้นที่ระดับตำบล

## 16. แนวคิดและประเภทของ JOIN

JOIN คือการเชื่อมข้อมูลจากสองตารางขึ้นไปด้วย Key ที่มีความหมายร่วมกัน

ประเภทหลัก ได้แก่

| JOIN | ผลลัพธ์ |
|---|---|
| `INNER JOIN` | เก็บเฉพาะ Key ที่พบในทั้งสองฝั่ง |
| `LEFT JOIN` | เก็บทุก Key จากตารางซ้าย |
| `RIGHT JOIN` | เก็บทุก Key จากตารางขวา |
| `FULL OUTER JOIN` | เก็บทุก Key จากทั้งสองฝั่ง |

การเลือก JOIN ต้องเริ่มจากคำถามวิเคราะห์

ตัวอย่างคำถามและ JOIN ที่เหมาะสม

| คำถาม | JOIN |
|---|---|
| พื้นที่ใดมีข้อมูลทั้ง Farmer และ MSO | `INNER JOIN` |
| เริ่มจากพื้นที่ Farmer และเติมข้อมูล MSO | `LEFT JOIN` |
| เริ่มจากพื้นที่ MSO และเติมข้อมูล Farmer | `RIGHT JOIN` |
| ต้องการตรวจ Coverage ของทั้งสองแหล่ง | `FULL OUTER JOIN` |

## 17. สรุปข้อมูลก่อน JOIN

สร้าง Summary ของ Farmer ที่ Grain ระดับพื้นที่

In [61]:
farmer_area_df = con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        COUNT(*) AS farmer_records,
        COUNT(DISTINCT pid)
            AS unique_farmers,
        AVG(total_income)
            AS avg_total_income,
        AVG(total_debt)
            AS avg_total_debt
    FROM farmer
    GROUP BY
        province,
        amphur,
        tambon
    """
).df()

farmer_area_df.head()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt
0,NaN,NaN,NaN,85,85,88165.625,39765.625
1,สุรินทร์,รัตนบุรี,เบิด,1,1,NaN,NaN
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,NaN,NaN
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,NaN,NaN
4,อุดรธานี,เพ็ญ,นาพู่,1,1,NaN,NaN


สร้าง Summary ของ MSO ที่ Grain เดียวกัน

In [62]:
mso_area_df = con.sql(
    """
    SELECT
        province,
        amphur,
        tambon,
        COUNT(*) AS mso_records,
        COUNT(DISTINCT household_id)
            AS unique_households,
        AVG(age) AS avg_age
    FROM mso
    GROUP BY
        province,
        amphur,
        tambon
    """
).df()

mso_area_df.head()

,province,amphur,tambon,mso_records,unique_households,avg_age
0,อุดรธานี,นายูง,โนนทอง,1,1,3.0
1,ราชบุรี,ดำเนินสะดวก,ท่านัด,1,1,55.0
2,หนองคาย,ท่าบ่อ,นาข่า,1,1,33.0
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,1,1,5.0
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,1,1,3.0


ตรวจสอบว่า Summary มีหนึ่งแถวต่อหนึ่ง Key จริงหรือไม่

In [63]:
farmer_area_duplicate_count = (
    farmer_area_df
    .duplicated(
        subset=area_columns
    )
    .sum()
)

farmer_area_duplicate_count

np.int64(0)

In [64]:
mso_area_duplicate_count = (
    mso_area_df
    .duplicated(
        subset=area_columns
    )
    .sum()
)

mso_area_duplicate_count

np.int64(0)

หากผลลัพธ์เป็น `0` แสดงว่าไม่มี Key ซ้ำใน Summary แต่ยังต้องตรวจค่าว่างและความสอดคล้องของชื่อพื้นที่ด้วย

## 18. ใช้ CTE จัดโครงสร้าง Query

CTE ย่อมาจาก **Common Table Expression**

สร้างด้วย `WITH` และใช้ตั้งชื่อผลลัพธ์ชั่วคราวภายใน Query

ข้อดี ได้แก่

- แบ่ง Query เป็นขั้นตอน
- อ่านง่าย
- ลด Subquery ซ้อนกัน
- ตรวจ Logic ได้ง่าย
- ใช้ผลลัพธ์ชั่วคราวหลายครั้งใน Query เดียว

In [65]:
con.sql(
    """
    WITH farmer_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS farmer_records,
            COUNT(DISTINCT pid)
                AS unique_farmers
        FROM farmer
        GROUP BY
            province,
            amphur,
            tambon
    )

    SELECT *
    FROM farmer_area
    ORDER BY farmer_records DESC
    LIMIT 10
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers
0,NaN,NaN,NaN,85,85
1,สุรินทร์,รัตนบุรี,เบิด,1,1
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1
4,แพร่,ร้องกวาง,แม่ยางร้อง,1,1
5,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1
6,อุดรธานี,เพ็ญ,นาพู่,1,1
7,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1


## 19. เชื่อมข้อมูลด้วย `INNER JOIN`

`INNER JOIN` เก็บเฉพาะพื้นที่ที่มี Key ตรงกันในทั้ง Farmer และ MSO

In [66]:
con.sql(
    """
    WITH farmer_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS farmer_records,
            COUNT(DISTINCT pid)
                AS unique_farmers
        FROM farmer
        GROUP BY
            province,
            amphur,
            tambon
    ),

    mso_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS mso_records,
            COUNT(DISTINCT household_id)
                AS unique_households
        FROM mso
        GROUP BY
            province,
            amphur,
            tambon
    )

    SELECT
        f.province,
        f.amphur,
        f.tambon,
        f.farmer_records,
        f.unique_farmers,
        m.mso_records,
        m.unique_households
    FROM farmer_area AS f
    INNER JOIN mso_area AS m
        ON f.province = m.province
        AND f.amphur = m.amphur
        AND f.tambon = m.tambon
    ORDER BY
        f.farmer_records DESC
    LIMIT 20
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,mso_records,unique_households


## 20. เชื่อมข้อมูลด้วย `LEFT JOIN`

`LEFT JOIN` เก็บทุกพื้นที่จากตารางซ้าย

ในตัวอย่างนี้ Farmer เป็นตารางหลัก

In [67]:
con.sql(
    """
    WITH farmer_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS farmer_records,
            COUNT(DISTINCT pid)
                AS unique_farmers,
            AVG(total_income)
                AS avg_total_income,
            AVG(total_debt)
                AS avg_total_debt
        FROM farmer
        GROUP BY
            province,
            amphur,
            tambon
    ),

    mso_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS mso_records,
            COUNT(DISTINCT household_id)
                AS unique_households,
            AVG(age) AS avg_age
        FROM mso
        GROUP BY
            province,
            amphur,
            tambon
    )

    SELECT
        f.province,
        f.amphur,
        f.tambon,
        f.farmer_records,
        f.unique_farmers,
        f.avg_total_income,
        f.avg_total_debt,
        m.mso_records,
        m.unique_households,
        m.avg_age
    FROM farmer_area AS f
    LEFT JOIN mso_area AS m
        ON f.province = m.province
        AND f.amphur = m.amphur
        AND f.tambon = m.tambon
    ORDER BY
        f.farmer_records DESC
    LIMIT 20
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age
0,NaN,NaN,NaN,85,85,88165.625,39765.625,<NA>,<NA>,NaN
1,สุรินทร์,รัตนบุรี,เบิด,1,1,NaN,NaN,<NA>,<NA>,NaN
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,NaN,NaN,<NA>,<NA>,NaN
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,NaN,NaN,<NA>,<NA>,NaN
4,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,NaN,NaN,<NA>,<NA>,NaN
5,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,NaN,NaN,<NA>,<NA>,NaN
6,อุดรธานี,เพ็ญ,นาพู่,1,1,NaN,NaN,<NA>,<NA>,NaN
7,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,NaN,NaN,<NA>,<NA>,NaN
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,NaN,NaN,<NA>,<NA>,NaN
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,NaN,NaN,<NA>,<NA>,NaN


## 21. เชื่อมข้อมูลด้วย `FULL OUTER JOIN`

`FULL OUTER JOIN` เก็บพื้นที่จากทั้งสองแหล่ง

เหมาะกับการตรวจ Coverage และ Key ที่เชื่อมไม่สำเร็จ

In [68]:
con.sql(
    """
    WITH farmer_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS farmer_records,
            COUNT(DISTINCT pid)
                AS unique_farmers
        FROM farmer
        GROUP BY
            province,
            amphur,
            tambon
    ),

    mso_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS mso_records,
            COUNT(DISTINCT household_id)
                AS unique_households
        FROM mso
        GROUP BY
            province,
            amphur,
            tambon
    )

    SELECT
        COALESCE(
            f.province,
            m.province
        ) AS province,
        COALESCE(
            f.amphur,
            m.amphur
        ) AS amphur,
        COALESCE(
            f.tambon,
            m.tambon
        ) AS tambon,
        f.farmer_records,
        f.unique_farmers,
        m.mso_records,
        m.unique_households,
        CASE
            WHEN
                f.province IS NOT NULL
                AND m.province IS NOT NULL
                THEN 'matched_both'
            WHEN
                f.province IS NOT NULL
                AND m.province IS NULL
                THEN 'farmer_only'
            WHEN
                f.province IS NULL
                AND m.province IS NOT NULL
                THEN 'mso_only'
        END AS match_status
    FROM farmer_area AS f
    FULL OUTER JOIN mso_area AS m
        ON f.province = m.province
        AND f.amphur = m.amphur
        AND f.tambon = m.tambon
    ORDER BY
        match_status,
        province,
        amphur,
        tambon
    LIMIT 30
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,mso_records,unique_households,match_status
0,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,<NA>,<NA>,farmer_only
1,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,<NA>,<NA>,farmer_only
2,พะเยา,ภูซาง,ทุ่งกล้วย,1,1,<NA>,<NA>,farmer_only
3,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,<NA>,<NA>,farmer_only
4,สุรินทร์,รัตนบุรี,เบิด,1,1,<NA>,<NA>,farmer_only
5,อุดรธานี,เพ็ญ,นาพู่,1,1,<NA>,<NA>,farmer_only
6,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,<NA>,<NA>,farmer_only
7,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,<NA>,<NA>,farmer_only
8,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,<NA>,<NA>,farmer_only
9,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,<NA>,<NA>,farmer_only


## 22. จัดการ `NULL` ด้วย `COALESCE`

`COALESCE()` คืนค่าตัวแรกที่ไม่เป็น `NULL`

รูปแบบคือ

```sql
COALESCE(value_1, value_2, ...)
```

ตัวอย่าง:

```sql
COALESCE(mso_records, 0)
```

หมายถึง หาก `mso_records` เป็น `NULL` ให้ใช้ `0`

ข้อควรระวัง:

`0` หลัง JOIN หมายถึง

> ไม่พบข้อมูลที่ Match ใน Dataset ปัจจุบัน

ไม่ได้ยืนยันว่าในโลกจริงไม่มีบุคคลหรือครัวเรือนในพื้นที่นั้น

In [69]:
con.sql(
    """
    WITH farmer_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS farmer_records,
            COUNT(DISTINCT pid)
                AS unique_farmers,
            AVG(total_income)
                AS avg_total_income,
            AVG(total_debt)
                AS avg_total_debt
        FROM farmer
        GROUP BY
            province,
            amphur,
            tambon
    ),

    mso_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS mso_records,
            COUNT(DISTINCT household_id)
                AS unique_households,
            AVG(age) AS avg_age
        FROM mso
        GROUP BY
            province,
            amphur,
            tambon
    ),

    joined_area AS (
        SELECT
            f.province,
            f.amphur,
            f.tambon,
            f.farmer_records,
            f.unique_farmers,
            f.avg_total_income,
            f.avg_total_debt,
            COALESCE(
                m.mso_records,
                0
            ) AS mso_records,
            COALESCE(
                m.unique_households,
                0
            ) AS unique_households,
            m.avg_age
        FROM farmer_area AS f
        LEFT JOIN mso_area AS m
            ON f.province = m.province
            AND f.amphur = m.amphur
            AND f.tambon = m.tambon
    )

    SELECT
        *,
        farmer_records + mso_records
            AS total_related_records,
        farmer_records - mso_records
            AS gap_score,
        CASE
            WHEN mso_records > 0
                THEN 'has_mso_data'
            ELSE 'farmer_only'
        END AS mso_match_status
    FROM joined_area
    ORDER BY
        total_related_records DESC
    LIMIT 20
    """
).df()

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age,total_related_records,gap_score,mso_match_status
0,NaN,NaN,NaN,85,85,88165.625,39765.625,0,0,NaN,85,85,farmer_only
1,สุรินทร์,รัตนบุรี,เบิด,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
4,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
5,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
6,อุดรธานี,เพ็ญ,นาพู่,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
7,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only


## 23. ตรวจสอบผลลัพธ์หลัง JOIN

หลัง JOIN ควรตรวจสอบอย่างน้อย

1. ตารางก่อน JOIN มีจำนวน Key เท่าใด
2. ผลลัพธ์หลัง JOIN มีจำนวนแถวเท่าใด
3. Key ในแต่ละฝั่งไม่ซ้ำตาม Grain หรือไม่
4. มีพื้นที่ Match ทั้งสองฝั่งกี่พื้นที่
5. มีพื้นที่ Farmer-only กี่พื้นที่
6. มีพื้นที่ MSO-only กี่พื้นที่
7. มี `NULL` ในคอลัมน์ใด
8. Key ที่ไม่ Match มีรูปแบบข้อความต่างกันหรือไม่

In [70]:
coverage_df = con.sql(
    """
    WITH farmer_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS farmer_records
        FROM farmer
        GROUP BY
            province,
            amphur,
            tambon
    ),

    mso_area AS (
        SELECT
            province,
            amphur,
            tambon,
            COUNT(*) AS mso_records
        FROM mso
        GROUP BY
            province,
            amphur,
            tambon
    ),

    coverage AS (
        SELECT
            COALESCE(
                f.province,
                m.province
            ) AS province,
            COALESCE(
                f.amphur,
                m.amphur
            ) AS amphur,
            COALESCE(
                f.tambon,
                m.tambon
            ) AS tambon,
            f.farmer_records,
            m.mso_records,
            CASE
                WHEN
                    f.province IS NOT NULL
                    AND m.province IS NOT NULL
                    THEN 'matched_both'
                WHEN
                    f.province IS NOT NULL
                    AND m.province IS NULL
                    THEN 'farmer_only'
                WHEN
                    f.province IS NULL
                    AND m.province IS NOT NULL
                    THEN 'mso_only'
            END AS match_status
        FROM farmer_area AS f
        FULL OUTER JOIN mso_area AS m
            ON f.province = m.province
            AND f.amphur = m.amphur
            AND f.tambon = m.tambon
    )

    SELECT *
    FROM coverage
    """
).df()

coverage_df.head()

,province,amphur,tambon,farmer_records,mso_records,match_status
0,อุดรธานี,นายูง,โนนทอง,<NA>,1,mso_only
1,ราชบุรี,ดำเนินสะดวก,ท่านัด,<NA>,1,mso_only
2,หนองคาย,ท่าบ่อ,นาข่า,<NA>,1,mso_only
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,<NA>,1,mso_only
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,<NA>,1,mso_only


In [71]:
coverage_df[
    "match_status"
].value_counts(
    dropna=False
)

match_status
mso_only       99
farmer_only    10
NaN             1
Name: count, dtype: int64

In [72]:
coverage_df.isna().sum()

province           1
amphur             1
tambon             1
farmer_records    99
mso_records       11
match_status       1
dtype: int64

### ตรวจสอบ Key ที่เชื่อมไม่สำเร็จ

In [73]:
coverage_df.loc[
    coverage_df["match_status"]
    != "matched_both"
].head(20)

,province,amphur,tambon,farmer_records,mso_records,match_status
0,อุดรธานี,นายูง,โนนทอง,<NA>,1,mso_only
1,ราชบุรี,ดำเนินสะดวก,ท่านัด,<NA>,1,mso_only
2,หนองคาย,ท่าบ่อ,นาข่า,<NA>,1,mso_only
3,อุบลราชธานี,น้ำยืน,โดมประดิษฐ์,<NA>,1,mso_only
4,นครราชสีมา,สีคิ้ว,หนองหญ้าขาว,<NA>,1,mso_only
5,ประจวบคีรีขันธ์,เมืองประจวบคีรีขันธ์,อ่าวน้อย,<NA>,1,mso_only
6,นครศรีธรรมราช,ปากพนัง,ปากพนังฝั่งตะวันออก,<NA>,1,mso_only
7,บุรีรัมย์,หนองหงส์,ไทยสามัคคี,<NA>,1,mso_only
8,เชียงใหม่,เทศบาลตำบลเมืองนะ,เมืองนะ,<NA>,1,mso_only
9,สุพรรณบุรี,บางปลาม้า,มะขามล้ม,<NA>,1,mso_only


Key ที่ไม่ Match อาจเกิดจาก

- ค่าว่าง
- ช่องว่างหรืออักขระพิเศษ
- คำว่า “อำเภอ”, “เขต” หรือ “ตำบล” ไม่เหมือนกัน
- การสะกดชื่อพื้นที่ต่างกัน
- ชื่อเก่ากับชื่อใหม่
- Coverage ของข้อมูลต่างกันจริง
- Grain ของข้อมูลไม่เท่ากัน

ไม่ควรสรุปทันทีว่าพื้นที่นั้นไม่มีข้อมูลจริง

## 24. เก็บ Query เป็น Python Function

เมื่อ Query ต้องใช้ซ้ำ ควรเก็บไว้ใน Function

ข้อดี ได้แก่

- ลดการ Copy Query
- เรียกใช้ซ้ำได้
- กำหนด Parameter ได้
- ควบคุม Output ให้เป็น DataFrame
- นำไปประกอบใน Workflow ได้

In [74]:
def build_area_join_summary(
    connection
):
    query = """
        WITH farmer_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS farmer_records,
                COUNT(DISTINCT pid)
                    AS unique_farmers,
                AVG(total_income)
                    AS avg_total_income,
                AVG(total_debt)
                    AS avg_total_debt
            FROM farmer
            GROUP BY
                province,
                amphur,
                tambon
        ),

        mso_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS mso_records,
                COUNT(
                    DISTINCT household_id
                ) AS unique_households,
                AVG(age) AS avg_age
            FROM mso
            GROUP BY
                province,
                amphur,
                tambon
        ),

        joined_area AS (
            SELECT
                f.province,
                f.amphur,
                f.tambon,
                f.farmer_records,
                f.unique_farmers,
                f.avg_total_income,
                f.avg_total_debt,
                COALESCE(
                    m.mso_records,
                    0
                ) AS mso_records,
                COALESCE(
                    m.unique_households,
                    0
                ) AS unique_households,
                m.avg_age
            FROM farmer_area AS f
            LEFT JOIN mso_area AS m
                ON f.province = m.province
                AND f.amphur = m.amphur
                AND f.tambon = m.tambon
        )

        SELECT
            *,
            farmer_records
                + mso_records
                AS total_related_records,
            farmer_records
                - mso_records
                AS gap_score,
            CASE
                WHEN mso_records > 0
                    THEN 'has_mso_data'
                ELSE 'farmer_only'
            END AS mso_match_status
        FROM joined_area
        ORDER BY
            total_related_records DESC
    """

    return connection.sql(
        query
    ).df()

In [75]:
area_join_summary_df = (
    build_area_join_summary(
        con
    )
)

area_join_summary_df.head(10)

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age,total_related_records,gap_score,mso_match_status
0,NaN,NaN,NaN,85,85,88165.625,39765.625,0,0,NaN,85,85,farmer_only
1,สุรินทร์,รัตนบุรี,เบิด,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
2,บุรีรัมย์,นางรอง,ก้านเหลือง,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
3,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
4,อุดรธานี,เพ็ญ,นาพู่,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
5,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
6,แพร่,ร้องกวาง,แม่ยางร้อง,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
7,อุบลราชธานี,ตระการพืชผล,สะพือ,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
8,ขอนแก่น,บ้านไผ่,ป่าปอ,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only
9,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only


## 25. Function แบบรับ Parameter

เมื่อผู้ใช้ต้องการผลลัพธ์เฉพาะบางจังหวัด สามารถส่งชื่อจังหวัดเข้า Function ได้

ควรใช้ Parameter Binding แทนการต่อข้อความโดยตรงเข้า SQL

ตัวอย่าง Placeholder ของ DuckDB คือ `?`

In [76]:
def build_area_join_summary_by_province(
    connection,
    province_name,
):
    query = """
        WITH farmer_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS farmer_records,
                COUNT(DISTINCT pid)
                    AS unique_farmers,
                AVG(total_income)
                    AS avg_total_income,
                AVG(total_debt)
                    AS avg_total_debt
            FROM farmer
            WHERE province = ?
            GROUP BY
                province,
                amphur,
                tambon
        ),

        mso_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS mso_records,
                COUNT(
                    DISTINCT household_id
                ) AS unique_households,
                AVG(age) AS avg_age
            FROM mso
            WHERE province = ?
            GROUP BY
                province,
                amphur,
                tambon
        ),

        joined_area AS (
            SELECT
                f.province,
                f.amphur,
                f.tambon,
                f.farmer_records,
                f.unique_farmers,
                f.avg_total_income,
                f.avg_total_debt,
                COALESCE(
                    m.mso_records,
                    0
                ) AS mso_records,
                COALESCE(
                    m.unique_households,
                    0
                ) AS unique_households,
                m.avg_age
            FROM farmer_area AS f
            LEFT JOIN mso_area AS m
                ON f.province = m.province
                AND f.amphur = m.amphur
                AND f.tambon = m.tambon
        )

        SELECT
            *,
            farmer_records
                + mso_records
                AS total_related_records,
            farmer_records
                - mso_records
                AS gap_score,
            CASE
                WHEN mso_records > 0
                    THEN 'has_mso_data'
                ELSE 'farmer_only'
            END AS mso_match_status
        FROM joined_area
        ORDER BY
            total_related_records DESC
    """

    parameters = [
        province_name,
        province_name,
    ]

    return connection.execute(
        query,
        parameters,
    ).df()

In [77]:
build_area_join_summary_by_province(
    con,
    "อุดรธานี",
).head(10)

,province,amphur,tambon,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,avg_age,total_related_records,gap_score,mso_match_status
0,อุดรธานี,เพ็ญ,นาพู่,1,1,NaN,NaN,0,0,NaN,1,1,farmer_only


# แบบฝึกหัดท้ายบท 
ให้ผู้เรียนใช้ table ที่ register ไว้แล้ว ได้แก่ 
- `farmer` 
- `mso` 

เพื่อสร้าง function สำหรับตอบคำถามวิเคราะห์จากสถานการณ์จริง

## แบบฝึกหัดที่ 1: สรุปข้อมูลเกษตรกรรายจังหวัด 

ผู้บริหารต้องการทราบว่าจังหวัดใดมีข้อมูลเกษตรกรเปราะบางมากที่สุด 

และแต่ละจังหวัดมีรายได้และหนี้สินเฉลี่ยเท่าใด ให้สร้าง function ชื่อ

```python 
def summarize_farmer_by_province(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

province
- farmer_records
- unique_farmers
- avg_total_income
- avg_total_debt

เงื่อนไข:

- ใช้ COUNT(*) สำหรับจำนวน record
- ใช้ COUNT(DISTINCT pid) สำหรับจำนวนบุคคลไม่ซ้ำ
- ใช้ AVG(total_income) สำหรับรายได้เฉลี่ย
- ใช้ AVG(total_debt) สำหรับหนี้สินเฉลี่ย
- เรียงจาก farmer_records มากไปน้อย

In [1]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 1
>
> Query ต้องจัดกลุ่มด้วย `province` และใช้ Aggregate Function ตามข้อกำหนด
>
> Function คืนผลลัพธ์เป็น pandas DataFrame ด้วย `.df()`

In [78]:
def summarize_farmer_by_province(
    con
):
    query = """
        SELECT
            province,
            COUNT(*)
                AS farmer_records,
            COUNT(DISTINCT pid)
                AS unique_farmers,
            AVG(total_income)
                AS avg_total_income,
            AVG(total_debt)
                AS avg_total_debt
        FROM farmer
        GROUP BY province
        ORDER BY
            farmer_records DESC
    """

    return con.sql(query).df()

In [79]:
farmer_province_summary = (
    summarize_farmer_by_province(
        con
    )
)

farmer_province_summary.head(10)

,province,farmer_records,unique_farmers,avg_total_income,avg_total_debt
0,NaN,85,85,88165.625,39765.625
1,อุบลราชธานี,3,3,NaN,NaN
2,อุดรธานี,1,1,NaN,NaN
3,แพร่,1,1,NaN,NaN
4,ศรีสะเกษ,1,1,NaN,NaN
5,พะเยา,1,1,NaN,NaN
6,บุรีรัมย์,1,1,NaN,NaN
7,ขอนแก่น,1,1,NaN,NaN
8,สุรินทร์,1,1,NaN,NaN


## แบบฝึกหัดที่ 2: สรุปข้อมูล MSO รายจังหวัด

ทีมวิเคราะห์ต้องการดูว่าจังหวัดใดมีข้อมูล logbook จาก MSO มากที่สุด และมีจำนวนครัวเรือนไม่ซ้ำเท่าใด

ให้สร้าง function ชื่อ

```python
def summarize_mso_by_province(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- mso_records
- unique_households
- avg_age

เงื่อนไข:

- สรุปข้อมูลระดับจังหวัด
- ใช้ COUNT(*) สำหรับจำนวน record
- ใช้ COUNT(DISTINCT household_id) สำหรับจำนวนครัวเรือนไม่ซ้ำ
- ใช้ AVG(age) สำหรับอายุเฉลี่ย
- เรียงจาก mso_records มากไปน้อย

In [2]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 2
>
> ใช้ `GROUP BY province` และแยกจำนวนแถวออกจากจำนวนครัวเรือนที่ไม่ซ้ำ

In [80]:
def summarize_mso_by_province(
    con
):
    query = """
        SELECT
            province,
            COUNT(*)
                AS mso_records,
            COUNT(
                DISTINCT household_id
            ) AS unique_households,
            AVG(age) AS avg_age
        FROM mso
        GROUP BY province
        ORDER BY
            mso_records DESC
    """

    return con.sql(query).df()

In [81]:
mso_province_summary = (
    summarize_mso_by_province(
        con
    )
)

mso_province_summary.head(10)

,province,mso_records,unique_households,avg_age
0,นครศรีธรรมราช,8,8,39.625000
1,อุดรธานี,7,7,27.142857
2,เชียงใหม่,5,5,30.600000
3,หนองคาย,4,4,36.500000
4,บุรีรัมย์,4,4,30.000000
5,นครนายก,4,4,36.750000
6,อุทัยธานี,4,4,53.000000
7,นครราชสีมา,4,4,61.000000
8,ขอนแก่น,4,4,70.500000
9,ชุมพร,3,3,56.333333


## แบบฝึกหัดที่ 3: เชื่อมข้อมูลระดับจังหวัดด้วย LEFT JOIN

ผู้บริหารต้องการเริ่มจากข้อมูลเกษตรกร และเติมข้อมูล MSO เข้ามา เพื่อดูว่าจังหวัดใดมีข้อมูลจากทั้งสองแหล่ง

ให้สร้าง function ชื่อ

```python
def build_province_left_join_summary(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- farmer_records
- unique_farmers
- avg_total_income
- avg_total_debt
- mso_records
- unique_households
- total_related_records
- mso_match_status

เงื่อนไข:

- ใช้ farmer เป็นข้อมูลหลัก
- ใช้ LEFT JOIN
- ถ้าไม่มีข้อมูลจาก MSO ให้แทน mso_records และ unique_households เป็น 0
- mso_match_status ให้มีค่าเป็น has_mso_data หรือ farmer_only

In [ ]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 3
>
> ควรสรุปทั้ง Farmer และ MSO ให้อยู่ในระดับจังหวัดก่อน JOIN
>
> ใช้ `COALESCE()` เปลี่ยน Metric ฝั่ง MSO ที่ไม่ Match จาก `NULL` เป็น `0`

In [82]:
def build_province_left_join_summary(
    con
):
    query = """
        WITH farmer_province AS (
            SELECT
                province,
                COUNT(*)
                    AS farmer_records,
                COUNT(DISTINCT pid)
                    AS unique_farmers,
                AVG(total_income)
                    AS avg_total_income,
                AVG(total_debt)
                    AS avg_total_debt
            FROM farmer
            GROUP BY province
        ),

        mso_province AS (
            SELECT
                province,
                COUNT(*)
                    AS mso_records,
                COUNT(
                    DISTINCT household_id
                ) AS unique_households
            FROM mso
            GROUP BY province
        ),

        joined_province AS (
            SELECT
                f.province,
                f.farmer_records,
                f.unique_farmers,
                f.avg_total_income,
                f.avg_total_debt,
                COALESCE(
                    m.mso_records,
                    0
                ) AS mso_records,
                COALESCE(
                    m.unique_households,
                    0
                ) AS unique_households
            FROM farmer_province AS f
            LEFT JOIN mso_province AS m
                ON f.province = m.province
        )

        SELECT
            *,
            farmer_records
                + mso_records
                AS total_related_records,
            CASE
                WHEN mso_records > 0
                    THEN 'has_mso_data'
                ELSE 'farmer_only'
            END AS mso_match_status
        FROM joined_province
        ORDER BY
            farmer_records DESC
    """

    return con.sql(query).df()

In [83]:
province_left_join_summary = (
    build_province_left_join_summary(
        con
    )
)

province_left_join_summary.head(10)

,province,farmer_records,unique_farmers,avg_total_income,avg_total_debt,mso_records,unique_households,total_related_records,mso_match_status
0,NaN,85,85,88165.625,39765.625,0,0,85,farmer_only
1,อุบลราชธานี,3,3,NaN,NaN,1,1,4,has_mso_data
2,อุดรธานี,1,1,NaN,NaN,7,7,8,has_mso_data
3,แพร่,1,1,NaN,NaN,1,1,2,has_mso_data
4,ศรีสะเกษ,1,1,NaN,NaN,1,1,2,has_mso_data
5,พะเยา,1,1,NaN,NaN,0,0,1,farmer_only
6,บุรีรัมย์,1,1,NaN,NaN,4,4,5,has_mso_data
7,ขอนแก่น,1,1,NaN,NaN,4,4,5,has_mso_data
8,สุรินทร์,1,1,NaN,NaN,0,0,1,farmer_only


## แบบฝึกหัดที่ 4: ตรวจสอบ Coverage ของข้อมูลด้วย FULL OUTER JOIN

ทีมข้อมูลต้องการตรวจสอบว่าพื้นที่ใดมีข้อมูลอยู่ในทั้งสองแหล่ง พื้นที่ใดมีเฉพาะ farmer และพื้นที่ใดมีเฉพาะ mso

ให้สร้าง function ชื่อ

```python
def check_area_coverage(con):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- amphur
- tambon
- farmer_records
- mso_records
- match_status

โดย match_status ต้องแบ่งเป็น

- matched_both
- farmer_only
- mso_only

เงื่อนไข:

- ใช้ FULL OUTER JOIN
- วิเคราะห์ในระดับ province + amphur + tambon
- ใช้ COALESCE เพื่อแสดงชื่อพื้นที่จากฝั่งที่มีข้อมูล

In [3]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 4
>
> สรุปแต่ละตารางให้อยู่ใน Grain `province + amphur + tambon` ก่อนใช้ `FULL OUTER JOIN`
>
> ใช้ `COALESCE()` กับชื่อพื้นที่เพื่อเลือกค่าจากฝั่งที่มีข้อมูล

In [84]:
def check_area_coverage(con):
    query = """
        WITH farmer_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS farmer_records
            FROM farmer
            GROUP BY
                province,
                amphur,
                tambon
        ),

        mso_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS mso_records
            FROM mso
            GROUP BY
                province,
                amphur,
                tambon
        )

        SELECT
            COALESCE(
                f.province,
                m.province
            ) AS province,
            COALESCE(
                f.amphur,
                m.amphur
            ) AS amphur,
            COALESCE(
                f.tambon,
                m.tambon
            ) AS tambon,
            f.farmer_records,
            m.mso_records,
            CASE
                WHEN
                    f.province IS NOT NULL
                    AND m.province IS NOT NULL
                    THEN 'matched_both'
                WHEN
                    f.province IS NOT NULL
                    AND m.province IS NULL
                    THEN 'farmer_only'
                WHEN
                    f.province IS NULL
                    AND m.province IS NOT NULL
                    THEN 'mso_only'
            END AS match_status
        FROM farmer_area AS f
        FULL OUTER JOIN mso_area AS m
            ON f.province = m.province
            AND f.amphur = m.amphur
            AND f.tambon = m.tambon
        ORDER BY
            match_status,
            province,
            amphur,
            tambon
    """

    return con.sql(query).df()

In [85]:
area_coverage_df = (
    check_area_coverage(
        con
    )
)

area_coverage_df.head(20)

,province,amphur,tambon,farmer_records,mso_records,match_status
0,ขอนแก่น,บ้านไผ่,ป่าปอ,1,<NA>,farmer_only
1,บุรีรัมย์,นางรอง,ก้านเหลือง,1,<NA>,farmer_only
2,พะเยา,ภูซาง,ทุ่งกล้วย,1,<NA>,farmer_only
3,ศรีสะเกษ,ขุขันธ์,ห้วยเหนือ,1,<NA>,farmer_only
4,สุรินทร์,รัตนบุรี,เบิด,1,<NA>,farmer_only
5,อุดรธานี,เพ็ญ,นาพู่,1,<NA>,farmer_only
6,อุบลราชธานี,ตระการพืชผล,สะพือ,1,<NA>,farmer_only
7,อุบลราชธานี,ตระการพืชผล,หนองเต่า,1,<NA>,farmer_only
8,อุบลราชธานี,ตระการพืชผล,ไหล่ทุ่ง,1,<NA>,farmer_only
9,แพร่,ร้องกวาง,แม่ยางร้อง,1,<NA>,farmer_only


## แบบฝึกหัดที่ 5: ค้นหาพื้นที่ที่มี farmer สูงแต่ mso ต่ำ

หน่วยงานต้องการค้นหาพื้นที่ที่อาจควรตรวจสอบเพิ่มเติม 
เพราะมีจำนวน record เกษตรกรสูง แต่มีข้อมูล MSO น้อยหรือไม่มีเลย

ให้สร้าง function ชื่อ

```python
def find_farmer_high_mso_low_area(con, min_farmer_records=10):
    ...
```

ผลลัพธ์ต้องมี column อย่างน้อย

- province
- amphur
- tambon
- farmer_records
- mso_records
- gap_score

กำหนดให้

> gap_score = farmer_records - mso_records

เงื่อนไข:

- ใช้ LEFT JOIN
- วิเคราะห์ในระดับ province + amphur + tambon
- แสดงเฉพาะพื้นที่ที่ farmer_records >= min_farmer_records
- เรียงจาก gap_score มากไปน้อย

> ### เฉลยแบบฝึกหัดที่ 5
>
> ใช้ Farmer เป็นตารางซ้าย และใช้ `COALESCE(mso_records, 0)` เพื่อให้พื้นที่ที่ไม่ Match สามารถคำนวณ `gap_score` ได้
>
> ใช้ Parameter Binding กับ `min_farmer_records` แทนการต่อค่าเข้า SQL โดยตรง

In [86]:
def find_farmer_high_mso_low_area(
    con,
    min_farmer_records=10,
):
    query = """
        WITH farmer_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS farmer_records
            FROM farmer
            GROUP BY
                province,
                amphur,
                tambon
        ),

        mso_area AS (
            SELECT
                province,
                amphur,
                tambon,
                COUNT(*)
                    AS mso_records
            FROM mso
            GROUP BY
                province,
                amphur,
                tambon
        ),

        joined_area AS (
            SELECT
                f.province,
                f.amphur,
                f.tambon,
                f.farmer_records,
                COALESCE(
                    m.mso_records,
                    0
                ) AS mso_records
            FROM farmer_area AS f
            LEFT JOIN mso_area AS m
                ON f.province = m.province
                AND f.amphur = m.amphur
                AND f.tambon = m.tambon
        )

        SELECT
            province,
            amphur,
            tambon,
            farmer_records,
            mso_records,
            farmer_records
                - mso_records
                AS gap_score
        FROM joined_area
        WHERE farmer_records >= ?
        ORDER BY
            gap_score DESC
    """

    return con.execute(
        query,
        [min_farmer_records],
    ).df()

In [87]:
farmer_high_mso_low_df = (
    find_farmer_high_mso_low_area(
        con,
        min_farmer_records=10,
    )
)

farmer_high_mso_low_df.head(20)

,province,amphur,tambon,farmer_records,mso_records,gap_score
0,None,None,None,85,0,85
